In [4]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv("drone_data.csv")


# ============================================================
# CONGESTION MODEL
# ============================================================

congestion_features = [
    "lane_id",
    "hour",
    "num_drones",
    "payload_kg",
    "wind_speed",
    "distance_km",
    "temperature",
    "visibility_km",
    "day_of_week",
    "battery_level",
    "drone_speed_kmh",
    "hub_distance_km",
    "is_peak_hour",
    "is_weekday"
]

X = df[congestion_features]
y = df["is_congested"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

congestion_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

congestion_model.fit(X_train, y_train)

y_pred = congestion_model.predict(X_test)

print("\n========== CONGESTION ==========")

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

joblib.dump(
    congestion_model,
    "congestion_model_new.pkl"
)

print("Congestion model saved.")




 



========== CONGESTION ==========
Accuracy: 0.951

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       844
           1       0.88      0.79      0.83       156

    accuracy                           0.95      1000
   macro avg       0.92      0.88      0.90      1000
weighted avg       0.95      0.95      0.95      1000

Congestion model saved.


In [7]:
"""
train_models.py

Training script for the AI Prediction module.

Models:
1. Congestion prediction
2. ETA prediction
3. Battery usage prediction

The training logic reproduces the training workflow used
to create the existing .pkl models.
"""

import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# CONFIGURATION
# ============================================================

DATA_PATH = "drone_data.csv"

MODEL_PATH = "congestion_model.pkl"
ETA_MODEL_PATH = "eta_model.pkl"
BATTERY_MODEL_PATH = "battery_model.pkl"

# IMPORTANT:
# The historical training notebook used 5 here.
# This value is required to reproduce the existing eta_model.pkl.
#
# The runtime config.py currently has MAX_DRONES_PER_LANE = 3.
# That is a runtime/project rule and is separate from the
# historical training formula used for the saved ETA model.
TRAINING_MAX_DRONES_PER_LANE = 5


# ============================================================
# LOAD DATA
# ============================================================

print("=" * 60)
print("LOADING DATA")
print("=" * 60)

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")


# ============================================================
# 1. CONGESTION MODEL
# ============================================================

print("\n" + "=" * 60)
print("CONGESTION MODEL")
print("=" * 60)


congestion_features = [
    "lane_id",
    "hour",
    "num_drones",
    "payload_kg",
    "wind_speed",
    "distance_km",
    "temperature",
    "visibility_km",
    "day_of_week",
    "battery_level",
    "drone_speed_kmh",
    "hub_distance_km",
    "is_peak_hour",
    "is_weekday"
]

congestion_target = "is_congested"


X_congestion = df[congestion_features]
y_congestion = df[congestion_target]


# 80% training / 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X_congestion,
    y_congestion,
    test_size=0.2,
    random_state=42
)


# Random Forest Classifier
congestion_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)


print("Training congestion model...")

congestion_model.fit(
    X_train,
    y_train
)


# Predictions
y_pred = congestion_model.predict(X_test)


# Evaluation
accuracy = accuracy_score(
    y_test,
    y_pred
)

print(f"\nAccuracy: {accuracy:.3f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)


# Feature importance
feature_importance = pd.Series(
    congestion_model.feature_importances_,
    index=congestion_features
).sort_values(ascending=False)

print("\nFeature Importances:")
print(feature_importance)


# Save model
joblib.dump(
    congestion_model,
    MODEL_PATH
)

print(f"\nCongestion model saved to: {MODEL_PATH}")


# ============================================================
# 2. CREATE ETA + BATTERY TARGETS
# ============================================================

print("\n" + "=" * 60)
print("CREATING ETA + BATTERY TARGETS")
print("=" * 60)


eta_features = [
    "distance_km",
    "wind_speed",
    "payload_kg",
    "drone_speed_kmh",
    "num_drones",
    "temperature",
    "visibility_km",
    "battery_level",
    "hour",
    "day_of_week"
]


# ------------------------------------------------------------
# ETA TARGET
# ------------------------------------------------------------

df["eta_minutes"] = (
    (
        df["distance_km"]
        / df["drone_speed_kmh"]
        * 60
    )
    *
    (
        1
        + df["wind_speed"] / 30
        + df["payload_kg"] / 10
        + (
            df["num_drones"]
            / TRAINING_MAX_DRONES_PER_LANE
            / 2
        )
    )
)

# Minimum ETA = 1 minute
df["eta_minutes"] = df["eta_minutes"].apply(
    lambda x: max(1, x)
)


# ------------------------------------------------------------
# BATTERY TARGET
# ------------------------------------------------------------

df["battery_used_percent"] = (
    (df["distance_km"] / 3)
    *
    (
        1
        + df["payload_kg"] / 5
        + df["wind_speed"] / 20
        + (50 - df["temperature"]) / 50
    )
)

# Keep battery consumption between 5% and 90%
df["battery_used_percent"] = (
    df["battery_used_percent"]
    .apply(
        lambda x: min(90, max(5, x))
    )
)


# ============================================================
# 3. ETA MODEL
# ============================================================

print("\n" + "=" * 60)
print("ETA MODEL")
print("=" * 60)


X_eta = df[eta_features]
y_eta = df["eta_minutes"]


X_eta_train, X_eta_test, y_eta_train, y_eta_test = (
    train_test_split(
        X_eta,
        y_eta,
        test_size=0.2,
        random_state=42
    )
)


eta_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)


print("Training ETA model...")

eta_model.fit(
    X_eta_train,
    y_eta_train
)


# Predictions
y_eta_pred = eta_model.predict(
    X_eta_test
)


# Evaluation
eta_mae = mean_absolute_error(
    y_eta_test,
    y_eta_pred
)

eta_rmse = np.sqrt(
    mean_squared_error(
        y_eta_test,
        y_eta_pred
    )
)

eta_r2 = r2_score(
    y_eta_test,
    y_eta_pred
)


print("\nETA Model Performance:")
print(f"MAE  : {eta_mae:.4f} minutes")
print(f"RMSE : {eta_rmse:.4f} minutes")
print(f"R2   : {eta_r2:.4f}")


# Save ETA model
joblib.dump(
    eta_model,
    ETA_MODEL_PATH
)

print(f"\nETA model saved to: {ETA_MODEL_PATH}")


# ============================================================
# 4. BATTERY MODEL
# ============================================================

print("\n" + "=" * 60)
print("BATTERY MODEL")
print("=" * 60)


X_battery = df[eta_features]
y_battery = df["battery_used_percent"]


X_bat_train, X_bat_test, y_bat_train, y_bat_test = (
    train_test_split(
        X_battery,
        y_battery,
        test_size=0.2,
        random_state=42
    )
)


battery_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)


print("Training battery model...")

battery_model.fit(
    X_bat_train,
    y_bat_train
)


# Predictions
y_bat_pred = battery_model.predict(
    X_bat_test
)


# Evaluation
battery_mae = mean_absolute_error(
    y_bat_test,
    y_bat_pred
)

battery_rmse = np.sqrt(
    mean_squared_error(
        y_bat_test,
        y_bat_pred
    )
)

battery_r2 = r2_score(
    y_bat_test,
    y_bat_pred
)


print("\nBattery Model Performance:")
print(f"MAE  : {battery_mae:.4f} %")
print(f"RMSE : {battery_rmse:.4f} %")
print(f"R2   : {battery_r2:.4f}")


# Save battery model
joblib.dump(
    battery_model,
    BATTERY_MODEL_PATH
)

print(
    f"\nBattery model saved to: "
    f"{BATTERY_MODEL_PATH}"
)


# ============================================================
# COMPLETE
# ============================================================

print("\n" + "=" * 60)
print("ALL THREE MODELS TRAINED SUCCESSFULLY")
print("=" * 60)

print("\nGenerated files:")
print(f"1. {MODEL_PATH}")
print(f"2. {ETA_MODEL_PATH}")
print(f"3. {BATTERY_MODEL_PATH}")


LOADING DATA
Dataset shape: (5000, 15)

CONGESTION MODEL
Training congestion model...

Accuracy: 0.951

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       844
           1       0.88      0.79      0.83       156

    accuracy                           0.95      1000
   macro avg       0.92      0.88      0.90      1000
weighted avg       0.95      0.95      0.95      1000


Feature Importances:
is_peak_hour       0.310141
num_drones         0.252373
hour               0.072439
drone_speed_kmh    0.057012
wind_speed         0.049743
payload_kg         0.043688
visibility_km      0.041568
battery_level      0.040473
distance_km        0.033433
hub_distance_km    0.030393
temperature        0.029873
day_of_week        0.016996
lane_id            0.016280
is_weekday         0.005588
dtype: float64

Congestion model saved to: congestion_model.pkl

CREATING ETA + BATTERY TARGETS

ETA MODEL
Training ETA model...

ET